In [1]:
import sys, random, os
from pathlib import Path
import pandas as pd
from selenium import webdriver
from time import sleep
sys.path.append('../')
from navigator import WebNavigatorHTML
from utils import format_path
from environments import WebWikiEnvironment

%load_ext autoreload
%autoreload 2

## Web navigator

In [ ]:
"""
--- ACTION REQUIRED ---
Gotta add here the path to the folder where the active html will be downloaded.
Can't be chosen : It must be the default downloads folder of the pc.
-----------------------
"""
downloads_folder = "/Users/nearchospotamitis/Downloads"

In [ ]:
paths = str(Path(os.getcwd()).parents[1]) + "/data/wikispeedia_paths-and-graph/paths_finished.tsv"
active_html = str(Path(os.getcwd()).parents[1]) + "/data/wpcd/wp/activeHTM/active.htm"

In [ ]:
def saveHTML(nav, path=active_html):
    """
    Saves the active HTML of the current navigator to a file.
    """
    html = nav.getHTML()
    with open(path, 'w+') as f:
        f.write(html)

def show_current_window(nav, window_driver ,path=active_html, wait_time=2):
    """
    Shows the current window of the navigator.
    """
    saveHTML(nav, active_html)
    window_driver.get("file://"+path)
    if wait_time:
        sleep(wait_time)

### Path traveling

In [ ]:
# Load paths
df = pd.read_csv(paths, sep="\t", skiprows=16, names = ["hashedIpAddress", "timestamp", "durationInSec", "path", "rating"])

In [ ]:
def travel_path(path, window, verbose=False, wait_time=None):
    """
    Algorithm to travel a path : Handles the navigator :
        - If the target is not in the page, move down until it is
        - If the target is in the page, move to the target
    """
    path = format_path(path)
    if verbose:
        print("Full path :\n"+"->".join(path) + "\n")

    path_position = 0
    current = path[path_position]
    target = path[path_position+1]
    final_target = path[-1]

    nav = WebNavigatorHTML(webdriver.Chrome(), pages=[current] ,downloads_folder=downloads_folder, markdown=True)
    show_current_window(nav, window, wait_time=wait_time)
    sleep(40)

    while current != final_target:
        show_current_window(nav, window, wait_time=wait_time)
    
        if target == "<":
            if verbose:
                print("{} <- {}".format(path[path_position-1], current))
            path_position -=1
            del path[path_position + 1]
            del path[path_position + 1]
            current = path[path_position]
            target = path[path_position + 1]
            nav.moveBack()

        else:
            links_in_page = nav.getLinks()

            while target not in links_in_page:
                nav.moveDown()
                show_current_window(nav, window, wait_time=wait_time)
                links_in_page = nav.getLinks()

            if verbose:
                print("{} -> {}".format(current, target))
            path_position +=1
            nav.movePage(target)
            current = path[path_position]
            if current == final_target:
                break
            else:
                target = path[path_position + 1]
    sleep(10)
    nav.quit()

# Driver showing active window
if "window" in locals():
    window.quit()
window = webdriver.Firefox()

# Load random path
#path = df.iloc[random.randint(0, len(df)-1)]["path"]

#Path : Cat;Ancient_Egypt;Engineering;Computer (Presentation)
path = df.iloc[2784]["path"]

# Travel path (wait_time : Time staying in an active window before moving on)f
travel_path(path, window=window, verbose=True, wait_time=1)
window.quit()

## Presentation

In [ ]:
if "nav" in locals():
    nav.quit()
if "window" in locals():
    window.quit()
nav = WebNavigatorHTML(webdriver.Chrome(), pages=["Cat"] ,downloads_folder=downloads_folder, markdown=True)
window = webdriver.Firefox()
show_current_window(nav, window, wait_time=0)

## Web Navigator WikiEnvironment

In [ ]:
"""
--- ACTION REQUIRED ---
Gotta add here the path to the folder where the active html will be downloaded.
Can't be chosen : It must be the default downloads folder of the pc.
-----------------------
"""
downloads_folder = "/Users/nearchospotamitis/Downloads"

In [ ]:
if "nav" in locals():
    nav.quit()
if "driver" in locals():
    driver.quit()

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--disable-gpu')  # Necessary for running in headless mode on some systems
options.add_argument('--no-sandbox') 
driver = webdriver.Chrome(options=options)

nav = WebNavigatorHTML(driver, pages=["Euro", "United_States", "Cyprus"] ,downloads_folder="", markdown=True, agents=["e", "u", "c"])


In [ ]:
all_headings = nav.getHeadings()
heading = random.choice(all_headings)
print("Looking for : '{}'".format(heading))

found = nav.moveHeading(heading)
print("Found : {}".format(found))
display(nav.df)

In [ ]:
nav.quit()

In [2]:
if "nav" in locals():
    nav.quit()
if "driver" in locals():
    driver.quit()
if "env" in locals():
    env.nav.quit()

# Setup driver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--disable-gpu')  # Necessary for running in headless mode on some systems
options.add_argument('--no-sandbox') 
driver = webdriver.Chrome(options=options)

# Setup navigator
nav = WebNavigatorHTML(driver, pages=["Euro", "United_States", "Cyprus"] ,downloads_folder="", markdown=True, agents=["e", "u", "c"])

# Setup environment
env = WebWikiEnvironment(current_page="Portuguese_language",nav = nav)

In [ ]:
env.nav.quit()